In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.build import load_config_defaults, load_predictor
from src.predict.postprocess import model_output_to_phase


In [ ]:
RUN_DIR = None
BATCH_SIZE = 8

run_dir = Path(RUN_DIR) if RUN_DIR else max(
    [p for p in (ROOT / "run").glob("*") if (p / "diffusion.yaml").is_file()],
    key=lambda p: p.stat().st_mtime,
)
run_dir = run_dir if run_dir.is_absolute() else ROOT / run_dir

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_phases = load_config_defaults(run_dir / "vae.yaml")["num_phases"]
predictor = load_predictor(run_dir, device=device)
vae = predictor.vae

shape = (BATCH_SIZE, int(vae.latent_ch), int(vae.latent_size), int(vae.latent_size))
with torch.no_grad():
    latent = predictor.sampler.sample(shape)
    generated = vae.decode(latent).detach().cpu().clamp(-1.0, 1.0)
phase = model_output_to_phase(generated, num_phases)

print("run dir:", run_dir)
print("generated:", tuple(generated.shape), "range:", (float(generated.min()), float(generated.max())))


In [ ]:
n = min(BATCH_SIZE, generated.shape[0])
rows = [("generated", generated[:n, 0], "gray", -1, 1), ("phase", phase[:n], "gray", 0, num_phases - 1)]

fig, axes = plt.subplots(len(rows), n, figsize=(n * 2, 4), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")
for r, (label, images, cmap, vmin, vmax) in enumerate(rows):
    axes[r, 0].text(-0.08, 0.5, label, transform=axes[r, 0].transAxes, ha="right", va="center")
    for c in range(n):
        axes[r, c].imshow(images[c], cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
plt.tight_layout()
